In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import h5py

In [2]:
torch.manual_seed(42)

In [3]:
device=torch.device("Cuda") if torch.cuda.is_available() else torch.device("cpu")
device

device(type='cpu')

In [4]:
train_set=h5py.File('../../data/Signs_Data_Training.h5')
test_set=h5py.File('../../data/Signs_Data_Testing.h5')

In [5]:
print(train_set.keys())
print(test_set.keys())

<KeysViewHDF5 ['list_classes', 'train_set_x', 'train_set_y']>
<KeysViewHDF5 ['list_classes', 'test_set_x', 'test_set_y']>


In [6]:
X_train = np.array(train_set['train_set_x'][:])
y_train = np.array(train_set['train_set_y'][:])

X_test = np.array(test_set['test_set_x'][:])
y_test = np.array(test_set['test_set_y'][:])

In [7]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (1080, 64, 64, 3)
y_train shape: (1080,)
X_test shape: (120, 64, 64, 3)
y_test shape: (120,)


In [8]:
print("Unique labels in y_test:", np.unique(y_test))

Unique labels in y_test: [0 1 2 3 4 5]


In [9]:
X_train = X_train/255.0
X_test = X_test/255.0

In [10]:
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):

  def __init__(self, features, labels,train=False):

    # Convert to PyTorch tensors
    self.features = torch.tensor(features, dtype=torch.float32).permute(0, 3, 1, 2)
    self.labels = torch.tensor(labels, dtype=torch.long)
    self.train = train
    
    self.transform=T.Compose([
            T.RandomHorizontalFlip(p=0.5),
            T.RandomRotation(degrees=15),
            T.RandomResizedCrop(size=(64, 64), scale=(0.85, 1.0)),
        ])
    
    

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    X=self.features[index]
    y=self.labels[index]
    
    if self.train:
      X=self.transform(X)
    return X,y
    

In [11]:
train_dataset = CustomDataset(X_train, y_train,train=True)
test_dataset = CustomDataset(X_test, y_test,train=True)

In [12]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [13]:
input_features=3

In [ ]:
import torch.nn as nn

class MyNN(nn.Module):
    def __init__(self, input_features):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(input_features,10,kernel_size=5,padding='same'),
            nn.BatchNorm2d(10),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2),
            nn.Conv2d(10,10,kernel_size=5,padding='same'),
            nn.BatchNorm2d(10),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2)
              
        )
        self.classifier_=nn.Sequential(
            nn.Flatten(),
            nn.Linear(10*16*16,100),
            nn.ReLU(),
            nn.Dropout(p=0.4),
            
            nn.Linear(100,50),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(50,6)
            
        )

    def forward(self, x):
        x=self.features(x)
        x=self.classifier_(x)
        return x

In [15]:
learning_rate = 0.01
epochs = 100

In [16]:
import torch.optim as optim

model = MyNN(3)

model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-4)

In [17]:
# training loop

for epoch in range(epochs):

  total_epoch_loss = 0

  for batch_features, batch_labels in train_loader:

    # move data to gpu
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    # forward pass
    outputs = model(batch_features)

    # calculate loss
    loss = criterion(outputs, batch_labels)

    # back pass
    optimizer.zero_grad()
    loss.backward()

    # update grads
    optimizer.step()

    total_epoch_loss = total_epoch_loss + loss.item()

  avg_loss = total_epoch_loss/len(train_loader)
  print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')


Epoch: 1 , Loss: 1.8024923661175896
Epoch: 2 , Loss: 1.785110775162192
Epoch: 3 , Loss: 1.776079822989071
Epoch: 4 , Loss: 1.7743038920795215
Epoch: 5 , Loss: 1.7647530471577364
Epoch: 6 , Loss: 1.753370628637426
Epoch: 7 , Loss: 1.7286252063863419
Epoch: 8 , Loss: 1.728063050438376
Epoch: 9 , Loss: 1.7103861009373384
Epoch: 10 , Loss: 1.67654007322648
Epoch: 11 , Loss: 1.6644555049784042
Epoch: 12 , Loss: 1.6175603656207813
Epoch: 13 , Loss: 1.5846783133114086
Epoch: 14 , Loss: 1.553813001688789
Epoch: 15 , Loss: 1.5027985713061165
Epoch: 16 , Loss: 1.4379649793400484
Epoch: 17 , Loss: 1.3875020952785717
Epoch: 18 , Loss: 1.3262098396525663
Epoch: 19 , Loss: 1.3172500063391293
Epoch: 20 , Loss: 1.2599460728028242
Epoch: 21 , Loss: 1.214039872674381
Epoch: 22 , Loss: 1.1794043989742504
Epoch: 23 , Loss: 1.132360412794001
Epoch: 24 , Loss: 1.1208720733137691
Epoch: 25 , Loss: 1.091142731554368
Epoch: 26 , Loss: 1.0719950374434977
Epoch: 27 , Loss: 1.0355922053841984
Epoch: 28 , Loss: 1.

In [18]:
model.eval()


MyNN(
  (features): Sequential(
    (0): Conv2d(3, 10, kernel_size=(5, 5), stride=(1, 1), padding=same)
    (1): BatchNorm2d(10, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(10, 10, kernel_size=(5, 5), stride=(1, 1), padding=same)
    (5): BatchNorm2d(10, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier_): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=2560, out_features=100, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=100, out_features=50, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.4, inplace=False)
    (7): Linear(in_features=50, out_features=6, bias=True)
  )
)

In [19]:
# evaluation on test data
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in test_loader:

    # move data to gpu
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    outputs = model(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.8666666666666667


In [20]:
# evaluation on training data
total = 0
correct = 0
with torch.no_grad():

  for batch_features, batch_labels in train_loader:

    # move data to gpu
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    outputs = model(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.8787037037037037
